In [1]:
import os
import re
import csv
import mysql.connector
from sshtunnel import SSHTunnelForwarder
import paramiko
import pandas as pd

def create_mysql_table(connection):
    cursor = connection.cursor()
    query = '''
    CREATE TABLE IF NOT EXISTS ExperimentData (
        experiment_batch VARCHAR(255) NOT NULL,
        experiment_id VARCHAR(255) NOT NULL,
        channel_id INT NOT NULL,
        heater_setting INT NOT NULL,
        timestamp INT NOT NULL,
        sensor_value FLOAT NOT NULL
    )
    '''
    cursor.execute(query)
    connection.commit()
    cursor.close()

def process_row(cursor, experiment_batch, experiment_id, channel_id, row):
    heater_setting, timestamp, sensor_value = row
    cursor.execute('''
    INSERT INTO ExperimentData (experiment_batch, experiment_id, channel_id, heater_setting, timestamp, sensor_value)
    VALUES (%s, %s, %s, %s, %s, %s)
    ''', (experiment_batch, experiment_id, channel_id, int(heater_setting), int(timestamp), float(sensor_value)))

def insert_data_from_csv(connection, experiment_batch, experiment_id, channel_id, file_path):
    cursor = connection.cursor()
    with open(file_path, 'r') as file:
        reader = csv.reader(file)
        first_row = next(reader)
        
        try:
            int(first_row[0])
            int(first_row[1])
            float(first_row[2])
            process_row(cursor, experiment_batch, experiment_id, channel_id, first_row)
        except ValueError:
            pass
        
        for row in reader:
            process_row(cursor, experiment_batch, experiment_id, channel_id, row)
    
    connection.commit()
    cursor.close()

def extract_channel_id(file_name):
    match = re.search(r'c(\d+)', file_name)
    if match:
        return int(match.group(1))
    return None

def process_folders(connection, root_folder):
    for batch_folder in os.listdir(root_folder):
        batch_folder_path = os.path.join(root_folder, batch_folder)
        if os.path.isdir(batch_folder_path):
            for csv_file in os.listdir(batch_folder_path):
                if csv_file.endswith('.csv') and '_BME680' not in csv_file:
                    csv_file_path = os.path.join(batch_folder_path, csv_file)
                    experiment_id = os.path.splitext(csv_file)[0]
                    channel_id = extract_channel_id(csv_file)
                    if channel_id is not None:
                        insert_data_from_csv(connection, batch_folder, experiment_id, channel_id, csv_file_path)

def main():
    ssh_config = paramiko.SSHConfig()
    ssh_config.parse(open('C:\\Users\\gavin\\.ssh\\config'))
    host = ssh_config.lookup('beagleServer')

    with SSHTunnelForwarder(
        (host['hostname'], 22),
        ssh_username=host['user'],
        ssh_pkey=host['identityfile'][0],
        remote_bind_address=('127.0.0.1', 3306)
    ) as tunnel:
        connection = mysql.connector.connect(
            host='127.0.0.1', 
            port=tunnel.local_bind_port,
            user='mysql-gas',
            password='mysql-gas',
            database='mysql-gas'
        )

        create_mysql_table(connection)

        # Root folder containing all batch folders
        root_folder = 'D:\\code\\uom_explore\\raw_data\\2024_07_29'
        process_folders(connection, root_folder)

        connection.close()

    print("Data upload complete")

if __name__ == "__main__":
    main()

Error uploading data to BME_Data: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '160, 1073798, 26273) VALUES (162, 1073809, 26215)' at line 1
Error uploading data to ExperimentData: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '1074154, 25.12, 69.07, 100557.00) VALUES (1074525.0, 25.09, 69.1, 100557.0)' at line 1
Data upload complete
